# 1.KMeans 기초

## 1.1. 라이브러리 및 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_iris

In [ ]:
iris_data = load_iris()
iris_data.feature_names

In [ ]:
# 보다 편리한 데이터 핸들링을 위해 컬럼값 변환
iris_df = pd.DataFrame(data=iris_data.data, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
iris_df.head()

## 1.2. KMeans

# 2.임의 Clustering 데이터 생성 후 알고리즘 테스트

## 2.1. 라이브러리 불러오기

In [ ]:
from sklearn.datasets import make_blobs

## 2.2. 데이터 생성

In [ ]:
X, y = make_blobs(                         
                  n_samples=200,            
                  n_features=2,             
                  centers=3,                
                  cluster_std=0.9,          
                  shuffle=True,             
                  random_state=0)

In [ ]:
print(X.shape, y.shape)

In [ ]:
X

In [ ]:
y

In [ ]:
cluster_df = pd.DataFrame(data=X,
                          columns=['ftr1', 'ftr2'])
cluster_df['target'] = y
cluster_df.head()

- 시각화

In [ ]:
marker0 = cluster_df[cluster_df['target']==0].index
marker1 = cluster_df[cluster_df['target']==1].index
marker2 = cluster_df[cluster_df['target']==2].index

plt.scatter(x=cluster_df.loc[marker0,'ftr1'], y=cluster_df.loc[marker0,'ftr2'], marker='s') # 사각형
plt.scatter(x=cluster_df.loc[marker1,'ftr1'], y=cluster_df.loc[marker1,'ftr2'], marker='o') # 원형
plt.scatter(x=cluster_df.loc[marker2,'ftr1'], y=cluster_df.loc[marker2,'ftr2'], marker='^') # 삼각형

plt.show()

## 2.3.KMeans

**1) 선언하기**

**2) 학습하기**

In [ ]:
# 시각화
marker0 = cluster_df[cluster_df['target']==0].index
marker1 = cluster_df[cluster_df['target']==1].index
marker2 = cluster_df[cluster_df['target']==2].index

plt.scatter(x=cluster_df.loc[marker0,'ftr1'], y=cluster_df.loc[marker0,'ftr2'], marker='s') # 사각형
plt.scatter(x=cluster_df.loc[marker1,'ftr1'], y=cluster_df.loc[marker1,'ftr2'], marker='o') # 원형
plt.scatter(x=cluster_df.loc[marker2,'ftr1'], y=cluster_df.loc[marker2,'ftr2'], marker='^') # 삼각형

plt.show()

In [ ]:
# Subplots 생성 (1행 2열)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 그래프에 필요한 설정값
titles = ['Scatter Plot by Target', 'Scatter Plot by Cluster']
columns = ['target', 'cluster']
markers = ['s', 'o', '^']
x_label = 'ftr1'
y_label = 'ftr2'

# 반복문으로 두 개의 서브플롯 생성
for i in range(2):
    for cluster_num in range(3):
        marker_index = cluster_df[cluster_df[columns[i]] == cluster_num].index
        axes[i].scatter(
            x=cluster_df.loc[marker_index, 'ftr1'],
            y=cluster_df.loc[marker_index, 'ftr2'],
            marker=markers[cluster_num],
            label=f'Cluster {cluster_num}'
        )
    axes[i].set_title(titles[i])
    axes[i].set_xlabel(x_label)
    axes[i].set_ylabel(y_label)
    axes[i].legend()

# 레이아웃 조정 및 그래프 출력
plt.tight_layout()
plt.show()


## 2.4. 실루엣 분석

In [ ]:
# %pip install yellowbrick
# %uv add yellowbrick

# 3.클러스터별 평균 실루엣 계수의 시각화
- https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html

In [ ]:
### 여러개의 클러스터링 갯수를 List로 입력 받아 각각의 실루엣 계수를 면적으로 시각화한 함수 작성
def visualize_silhouette(cluster_list, X_features):

    import matplotlib.cm as cm
    import math
    import matplotlib.pyplot as plt
    import numpy as np

    from sklearn.cluster import KMeans
    from sklearn.datasets import make_blobs
    from sklearn.metrics import silhouette_samples, silhouette_score

    # 입력값으로 클러스터링 갯수들을 리스트로 받아서, 각 갯수별로 클러스터링을 적용하고 실루엣 개수를 구함
    n_cols = len(cluster_list)

    # plt.subplots()으로 리스트에 기재된 클러스터링 수만큼의 sub figures를 가지는 axs 생성
    fig, axs = plt.subplots(figsize=(4*n_cols, 4), nrows=1, ncols=n_cols)

    # 리스트에 기재된 클러스터링 갯수들을 차례로 iteration 수행하면서 실루엣 개수 시각화
    for ind, n_cluster in enumerate(cluster_list):

        # KMeans 클러스터링 수행하고, 실루엣 스코어와 개별 데이터의 실루엣 값 계산.
        clusterer = KMeans(n_clusters = n_cluster, max_iter=500, random_state=0)
        clusterer.fit(X_features)
        cluster_labels = clusterer.predict(X_features)

        sil_avg = silhouette_score(X_features, cluster_labels)
        sil_values = silhouette_samples(X_features, cluster_labels)

        y_lower = 10
        axs[ind].set_title('Number of Cluster : '+ str(n_cluster)+'\n' \
                          'Silhouette Score :' + str(round(sil_avg,3)) )
        axs[ind].set_xlabel("The silhouette coefficient values")
        axs[ind].set_ylabel("Cluster label")
        axs[ind].set_xlim([-0.1, 1])
        axs[ind].set_ylim([0, len(X_features) + (n_cluster + 1) * 10])
        axs[ind].set_yticks([])  # Clear the yaxis labels / ticks
        axs[ind].set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1])

        # 클러스터링 갯수별로 fill_betweenx( )형태의 막대 그래프 표현.
        for i in range(n_cluster):
            ith_cluster_sil_values = sil_values[cluster_labels==i]
            ith_cluster_sil_values.sort()

            size_cluster_i = ith_cluster_sil_values.shape[0]
            y_upper = y_lower + size_cluster_i

            color = cm.nipy_spectral(float(i) / n_cluster)
            axs[ind].fill_betweenx(np.arange(y_lower, y_upper),
                                   0,
                                   ith_cluster_sil_values,
                                   facecolor=color,
                                   edgecolor=color,
                                   alpha=0.7)
            axs[ind].text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
            y_lower = y_upper + 10

        axs[ind].axvline(x=sil_avg, color="red", linestyle="--")

In [ ]:
# make_blobs를 통해 군집화를 위한 4개의 군집 중심의 500개 2차원 데이터 세트 생성
X, y = make_blobs(n_samples=500, 
                  n_features=2, 
                  centers=4, 
                  cluster_std=1,
                  center_box=(-10.90, 10.0),
                  shuffle=True,
                  random_state=1)